# ⚡ Project 003 — Energy Consumption Optimizer · SME
**Portfolio project | Ishan Sewnandan**

Clustering Dutch SME energy consumption data to detect waste patterns and reduce costs.  
Tools: Python · Pandas · Scikit-learn (K-means) · Plotly · Streamlit

---
### Doel
- Energieverbruik van Nederlandse MKB-bedrijven per sector analyseren (elektriciteit + gas)
- K-means clustering toepassen om verbruiksprofielen te identificeren
- Verspilling detecteren per cluster en besparingspotentieel kwantificeren
- Interactief Streamlit dashboard bouwen voor MKB-advies

### Databronnen
- **CBS `84901NED`** — Aardgas- & elektriciteitslevering aan bedrijven; verbruiksklasse, SBI 2008
- **CBS `83989NED`** — Energiebalans; aanbod en verbruik per sector


## 0. Setup & Installatie

In [ ]:
# !pip install cbsodata pandas numpy scikit-learn plotly streamlit --break-system-packages

import cbsodata
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print('✅ Alle packages geladen')

## 1. Data Ophalen via CBS Open Data API

In [ ]:
# ─── Dataset 1: Energielevering aan bedrijven per sector & verbruiksklasse ────
print('📥 Dataset 1 ophalen: aardgas & elektriciteitslevering aan bedrijven...')
df_raw = pd.DataFrame(cbsodata.get_data('84901NED'))
print(f'   → {len(df_raw):,} rijen, {df_raw.shape[1]} kolommen')
print('Kolommen:', list(df_raw.columns))
df_raw.head(3)

In [ ]:
# ─── Dataset 2: Energiebalans per sector ─────────────────────────────────────
print('📥 Dataset 2 ophalen: energiebalans per sector...')
df_balans_raw = pd.DataFrame(cbsodata.get_data('83989NED'))
print(f'   → {len(df_balans_raw):,} rijen, {df_balans_raw.shape[1]} kolommen')
df_balans_raw.head(3)

## 2. Data Cleaning & Exploratie

In [ ]:
# ─── Kolomnamen inspecteren ───────────────────────────────────────────────────
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]

print('Alle kolomnamen en voorbeeldwaarden:')
for col in df.columns:
    print(f'  [{col}] dtype={df[col].dtype} | voorbeeld: {df[col].iloc[0]}')

In [ ]:
# ─── Relevante kolommen identificeren ────────────────────────────────────────
# CBS 84901NED bevat: sector (SBI), verbruiksklasse, aantal adressen,
# totale elektriciteitslevering (kWh), totale aardgaslevering (m3), jaar

# Detecteer automatisch de juiste kolomnamen
sector_col   = [c for c in df.columns if 'bedrijfstak' in c.lower() or 'sbi' in c.lower() or 'sector' in c.lower()]
klasse_col   = [c for c in df.columns if 'klasse' in c.lower() or 'verbruik' in c.lower()]
elek_col     = [c for c in df.columns if 'elektriciteit' in c.lower() or 'elek' in c.lower()]
gas_col      = [c for c in df.columns if 'aardgas' in c.lower() or 'gas' in c.lower()]
periode_col  = [c for c in df.columns if 'periode' in c.lower() or 'jaar' in c.lower()]
adressen_col = [c for c in df.columns if 'adres' in c.lower() or 'aantal' in c.lower()]

print('Gevonden kolomgroepen:')
print(f'  Sector:    {sector_col}')
print(f'  Klasse:    {klasse_col}')
print(f'  Elektra:   {elek_col}')
print(f'  Gas:       {gas_col}')
print(f'  Periode:   {periode_col}')
print(f'  Adressen:  {adressen_col}')

In [ ]:
# ─── Dataset opschonen ───────────────────────────────────────────────────────
# Gebruik eerste gevonden kolom van elke groep (aanpassen indien nodig)
COL_SECTOR  = sector_col[0]   if sector_col   else 'BedrijfstakkenSBI2008'
COL_KLASSE  = klasse_col[0]   if klasse_col   else 'Verbruiksklasse'
COL_ELEK    = elek_col[0]     if elek_col     else 'ElektriciteitskWh'
COL_GAS     = gas_col[0]      if gas_col      else 'AardgasM3'
COL_PERIODE = periode_col[0]  if periode_col  else 'Perioden'
COL_ADR     = adressen_col[0] if adressen_col else 'AantalAdressen'

# Numerieke kolommen converteren
for col in [COL_ELEK, COL_GAS, COL_ADR]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Filter op meest recente jaar
if COL_PERIODE in df.columns:
    df['jaar'] = df[COL_PERIODE].astype(str).str[:4]
    df['jaar'] = pd.to_numeric(df['jaar'], errors='coerce')
    meest_recent = df['jaar'].max()
    df_recent = df[df['jaar'] == meest_recent].copy()
    print(f'Meest recente jaar: {meest_recent} | {len(df_recent):,} rijen')
else:
    df_recent = df.copy()

df_recent = df_recent.dropna(subset=[COL_ELEK, COL_GAS])
print(f'Na cleaning: {len(df_recent):,} rijen')
df_recent.head()

In [ ]:
# ─── MKB-relevant: filter kleine & middelgrote verbruiksklassen ──────────────
# CBS verbruiksklassen gaan van klein (<10k kWh) tot groot (>50M kWh)
# MKB = klassen 1 t/m 4 (ruwweg <1 MWh tot ~50MWh elektra)
print('Unieke verbruiksklassen:')
print(df_recent[COL_KLASSE].unique())

# Filter MKB-klassen (pas aan op basis van output hierboven)
# Typische CBS klasse-codes: 'A', 'B', 'C', 'D' of numeriek
# Als klasse strings zijn, filter op alles dat niet 'groot' of 'industrie' bevat
mkb_mask = ~df_recent[COL_KLASSE].astype(str).str.lower().str.contains(
    'groot|large|>50|>100|5000|10000|industri', na=False
)
df_mkb = df_recent[mkb_mask].copy()
print(f'\nMKB-rijen: {len(df_mkb):,} (van {len(df_recent):,} totaal)')

In [ ]:
# ─── Gemiddeld verbruik per adres berekenen ───────────────────────────────────
# Dit maakt clusters vergelijkbaar ongeacht sector-omvang
if COL_ADR in df_mkb.columns and df_mkb[COL_ADR].sum() > 0:
    df_mkb['gem_elek_kwh']  = df_mkb[COL_ELEK] / df_mkb[COL_ADR].replace(0, np.nan) * 1000
    df_mkb['gem_gas_m3']    = df_mkb[COL_GAS]  / df_mkb[COL_ADR].replace(0, np.nan) * 1000
else:
    df_mkb['gem_elek_kwh'] = df_mkb[COL_ELEK]
    df_mkb['gem_gas_m3']   = df_mkb[COL_GAS]

# Totaal energieverbruik in kWh-equivalent (gas: 1 m3 ≈ 9.77 kWh)
df_mkb['totaal_kwh_eq'] = df_mkb['gem_elek_kwh'] + (df_mkb['gem_gas_m3'] * 9.77)

# Gas/elektra ratio (indicator voor energiemix)
df_mkb['gas_elek_ratio'] = df_mkb['gem_gas_m3'] / df_mkb['gem_elek_kwh'].replace(0, np.nan)

df_mkb = df_mkb.replace([np.inf, -np.inf], np.nan).dropna(subset=['gem_elek_kwh', 'gem_gas_m3'])
print(f'Dataset voor clustering: {len(df_mkb):,} rijen')
df_mkb[['gem_elek_kwh', 'gem_gas_m3', 'totaal_kwh_eq', 'gas_elek_ratio']].describe()

## 3. EDA — Verbruikspatronen per Sector

In [ ]:
# ─── Plot 1: Elektra vs Gas per sector ───────────────────────────────────────
fig1 = px.scatter(
    df_mkb.dropna(subset=[COL_SECTOR]),
    x='gem_elek_kwh', y='gem_gas_m3',
    color=COL_SECTOR,
    size='totaal_kwh_eq',
    title='⚡ Elektriciteits- vs Gasverbruik per MKB-sector',
    labels={
        'gem_elek_kwh': 'Gem. elektriciteitsverbruik (kWh/adres)',
        'gem_gas_m3': 'Gem. gasverbruik (m3/adres)'
    },
    template='plotly_white',
    hover_name=COL_SECTOR
)
fig1.update_layout(font_family='Georgia', title_font_size=16)
fig1.show()

In [ ]:
# ─── Plot 2: Top 10 sectoren op totaalverbruik ────────────────────────────────
sector_agg = df_mkb.groupby(COL_SECTOR)[['gem_elek_kwh', 'gem_gas_m3', 'totaal_kwh_eq']].mean()
sector_agg = sector_agg.nlargest(10, 'totaal_kwh_eq').reset_index()

fig2 = go.Figure()
fig2.add_trace(go.Bar(name='Elektriciteit (kWh)', x=sector_agg[COL_SECTOR],
                       y=sector_agg['gem_elek_kwh'], marker_color='#2563EB'))
fig2.add_trace(go.Bar(name='Aardgas (m3 × 9.77)', x=sector_agg[COL_SECTOR],
                       y=sector_agg['gem_gas_m3'] * 9.77, marker_color='#5C8A3C'))
fig2.update_layout(
    barmode='stack',
    title='🏭 Top 10 MKB-sectoren op energieverbruik (kWh-equivalent)',
    xaxis_tickangle=-35,
    template='plotly_white',
    font_family='Georgia',
    yaxis_title='kWh-equivalent per adres'
)
fig2.show()

## 4. K-means Clustering

In [ ]:
# ─── Features voor clustering ─────────────────────────────────────────────────
feature_cols = ['gem_elek_kwh', 'gem_gas_m3', 'totaal_kwh_eq', 'gas_elek_ratio']
X = df_mkb[feature_cols].dropna()

# Standaardiseren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Clustering op {len(X)} datapunten, {len(feature_cols)} features')

In [ ]:
# ─── Elbow method + Silhouette score ─────────────────────────────────────────
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

# Plot elbow + silhouette
fig3 = make_subplots(rows=1, cols=2, subplot_titles=('Elbow Method (Inertia)', 'Silhouette Score'))
fig3.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
                           name='Inertia', line=dict(color='#2563EB', width=2.5)), row=1, col=1)
fig3.add_trace(go.Scatter(x=list(K_range), y=silhouettes, mode='lines+markers',
                           name='Silhouette', line=dict(color='#5C8A3C', width=2.5)), row=1, col=2)
fig3.update_layout(title='🔍 Optimaal aantal clusters bepalen',
                   template='plotly_white', font_family='Georgia', showlegend=False)
fig3.show()

best_k = list(K_range)[silhouettes.index(max(silhouettes))]
print(f'\n✅ Beste k op basis van silhouette: k={best_k} (score={max(silhouettes):.3f})')

In [ ]:
# ─── Finaal K-means model trainen ─────────────────────────────────────────────
# Gebruik k=4 als typisch zinvol voor MKB-segmentatie
# (pas aan op basis van elbow/silhouette output hierboven)
K_FINAL = best_k if 3 <= best_k <= 6 else 4

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
df_mkb_clean = df_mkb[feature_cols].dropna().copy()
df_mkb_clean['cluster'] = kmeans.fit_predict(X_scaled)

# Sector toevoegen
df_mkb_clean[COL_SECTOR] = df_mkb.loc[df_mkb_clean.index, COL_SECTOR].values

print(f'\n✅ K-means klaar met k={K_FINAL}')
print('\nClustergrootte:')
print(df_mkb_clean['cluster'].value_counts().sort_index())

In [ ]:
# ─── Cluster profielen ────────────────────────────────────────────────────────
cluster_profiles = df_mkb_clean.groupby('cluster')[feature_cols].mean().round(0)

# Labels toewijzen op basis van verbruiksprofiel
cluster_profiles['totaal_rank'] = cluster_profiles['totaal_kwh_eq'].rank()

label_map = {}
for idx, row in cluster_profiles.iterrows():
    rank = int(row['totaal_rank'])
    gas_ratio = row['gas_elek_ratio']
    if rank == 1:
        label = '🟢 Laag verbruik · Efficiënt'
    elif rank == K_FINAL:
        label = '🔴 Hoog verbruik · Verbetering mogelijk'
    elif gas_ratio > cluster_profiles['gas_elek_ratio'].median():
        label = '🟡 Gas-intensief · Warmteverbruik'
    else:
        label = '🟠 Elektra-intensief · Procesverbruik'
    label_map[idx] = label

df_mkb_clean['cluster_label'] = df_mkb_clean['cluster'].map(label_map)

print('Cluster profielen:')
for cid, label in label_map.items():
    row = cluster_profiles.loc[cid]
    print(f'  Cluster {cid} — {label}')
    print(f'    Gem. elektra: {row["gem_elek_kwh"]:,.0f} kWh/adres')
    print(f'    Gem. gas:     {row["gem_gas_m3"]:,.0f} m3/adres')
    print(f'    Totaal:       {row["totaal_kwh_eq"]:,.0f} kWh-eq/adres')

In [ ]:
# ─── PCA voor 2D visualisatie ─────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_mkb_clean['pca_1'] = X_pca[:, 0]
df_mkb_clean['pca_2'] = X_pca[:, 1]

print(f'PCA verklaarde variantie: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# ─── Plot 4: Cluster scatter (PCA) ───────────────────────────────────────────
color_map = {
    0: '#5C8A3C', 1: '#F59E0B', 2: '#EF4444', 3: '#3B82F6',
    4: '#8B5CF6', 5: '#06B6D4'
}

fig4 = px.scatter(
    df_mkb_clean,
    x='pca_1', y='pca_2',
    color='cluster_label',
    hover_name=COL_SECTOR,
    hover_data={'gem_elek_kwh': ':,.0f', 'gem_gas_m3': ':,.0f', 'pca_1': False, 'pca_2': False},
    title=f'🔍 K-means Clusters (k={K_FINAL}) — MKB Energieverbruik · PCA visualisatie',
    labels={'pca_1': 'PCA component 1', 'pca_2': 'PCA component 2', 'cluster_label': 'Cluster'},
    template='plotly_white',
    opacity=0.8
)
fig4.update_traces(marker_size=9)
fig4.update_layout(font_family='Georgia', title_font_size=16)
fig4.show()
fig4.write_html('energy_clusters.html')
print('✅ Opgeslagen als energy_clusters.html')

## 5. Besparingspotentieel Analyse

In [ ]:
# ─── Benchmarking: verbruik t.o.v. het efficiëntste cluster ──────────────────
# Het laagst-verbrukende cluster is de benchmark
benchmark_cluster = cluster_profiles['totaal_kwh_eq'].idxmin()
benchmark_elek = cluster_profiles.loc[benchmark_cluster, 'gem_elek_kwh']
benchmark_gas  = cluster_profiles.loc[benchmark_cluster, 'gem_gas_m3']

# Energieprijzen NL MKB (2024 gemiddeld)
PRIJS_ELEK = 0.28   # €/kWh (MKB variabel tarief)
PRIJS_GAS  = 1.05   # €/m3  (MKB variabel tarief)

savings = []
for cid, label in label_map.items():
    row  = cluster_profiles.loc[cid]
    n    = (df_mkb_clean['cluster'] == cid).sum()
    
    overschot_elek = max(0, row['gem_elek_kwh'] - benchmark_elek)
    overschot_gas  = max(0, row['gem_gas_m3']   - benchmark_gas)
    besparing_eur  = (overschot_elek * PRIJS_ELEK) + (overschot_gas * PRIJS_GAS)
    
    savings.append({
        'cluster': cid,
        'label': label,
        'n_bedrijven': n,
        'gem_elek': row['gem_elek_kwh'],
        'gem_gas':  row['gem_gas_m3'],
        'overschot_elek_kwh': overschot_elek,
        'overschot_gas_m3':   overschot_gas,
        'besparing_eur_adres': besparing_eur,
        'besparing_totaal_eur': besparing_eur * n
    })

df_savings = pd.DataFrame(savings)
print('💰 Besparingspotentieel per cluster:')
print(df_savings[['label', 'n_bedrijven', 'besparing_eur_adres', 'besparing_totaal_eur']]
      .to_string(index=False))

In [ ]:
# ─── Plot 5: Besparingspotentieel per cluster ─────────────────────────────────
df_savings_plot = df_savings[df_savings['besparing_eur_adres'] > 0]

fig5 = px.bar(
    df_savings_plot.sort_values('besparing_eur_adres', ascending=False),
    x='label', y='besparing_eur_adres',
    color='besparing_eur_adres',
    color_continuous_scale='RdYlGn_r',
    title='💰 Geschat Besparingspotentieel per Cluster (€/adres/jaar)',
    labels={'label': 'Cluster', 'besparing_eur_adres': 'Besparing (€/adres)'},
    template='plotly_white'
)
fig5.update_layout(font_family='Georgia', showlegend=False,
                   yaxis_tickprefix='€', yaxis_tickformat=',.0f')
fig5.show()
fig5.write_html('energy_savings.html')
print('✅ Opgeslagen als energy_savings.html')

In [ ]:
# ─── Plot 6: Radar chart per cluster ─────────────────────────────────────────
categories = ['Elektra\n(kWh)', 'Gas\n(m3)', 'Totaal\n(kWh-eq)', 'Gas/Elek\nRatio']
cluster_norm = cluster_profiles[feature_cols].div(cluster_profiles[feature_cols].max())

fig6 = go.Figure()
colors = ['#5C8A3C', '#F59E0B', '#EF4444', '#3B82F6', '#8B5CF6']

for i, cid in enumerate(cluster_norm.index):
    vals = cluster_norm.loc[cid].tolist()
    vals += [vals[0]]  # sluiten
    fig6.add_trace(go.Scatterpolar(
        r=vals,
        theta=['Elektra', 'Gas', 'Totaal', 'Gas/Elek', 'Elektra'],
        fill='toself',
        name=label_map.get(cid, f'Cluster {cid}'),
        line_color=colors[i % len(colors)],
        opacity=0.65
    ))

fig6.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='🕸 Energieprofiel per Cluster (genormaliseerd)',
    template='plotly_white',
    font_family='Georgia'
)
fig6.show()

## 6. Streamlit Dashboard — Code

In [ ]:
# ─── Streamlit app code (sla op als app.py en run: streamlit run app.py) ─────
streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import cbsodata

st.set_page_config(
    page_title="⚡ Energy Optimizer · MKB NL",
    page_icon="⚡",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .main { background-color: #F4F2EE; }
    .stMetricValue { font-size: 2rem !important; }
    h1 { font-weight: 800; }
</style>
""", unsafe_allow_html=True)

# ── Header ────────────────────────────────────────────────────────────────────
st.title("⚡ Energy Consumption Optimizer")
st.markdown("**MKB Nederland** · Verbruiksprofielen detecteren & besparingen berekenen")
st.divider()

# ── Data laden ────────────────────────────────────────────────────────────────
@st.cache_data(ttl=3600)
def load_data():
    df = pd.DataFrame(cbsodata.get_data("84901NED"))
    return df

with st.spinner("CBS data ophalen..."):
    df_raw = load_data()

st.success(f"✅ {len(df_raw):,} records geladen van CBS StatLine")

# ── Sidebar: instellingen ─────────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Instellingen")
    k_clusters = st.slider("Aantal clusters (k)", min_value=2, max_value=8, value=4)
    prijs_elek  = st.number_input("Elektriciteitsprijs (€/kWh)", value=0.28, step=0.01)
    prijs_gas   = st.number_input("Gasprijs (€/m3)", value=1.05, step=0.05)
    st.divider()
    st.caption("Data: CBS StatLine · 84901NED")
    st.caption("Ishan Sewnandan · Portfolio 003")

# ── Data prep (simplified) ───────────────────────────────────────────────────
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]

# Kolom detectie
elek_col   = next((c for c in df.columns if "elektriciteit" in c.lower()), None)
gas_col    = next((c for c in df.columns if "aardgas" in c.lower()), None)
sector_col = next((c for c in df.columns if "bedrijfstak" in c.lower() or "sbi" in c.lower()), None)
adr_col    = next((c for c in df.columns if "adres" in c.lower()), None)

if not all([elek_col, gas_col]):
    st.error("Kolomnamen anders dan verwacht. Pas de detectie aan in de notebook.")
    st.stop()

for col in [elek_col, gas_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
if adr_col:
    df[adr_col] = pd.to_numeric(df[adr_col], errors="coerce")
    df["gem_elek"] = df[elek_col] / df[adr_col].replace(0, np.nan) * 1000
    df["gem_gas"]  = df[gas_col]  / df[adr_col].replace(0, np.nan) * 1000
else:
    df["gem_elek"] = df[elek_col]
    df["gem_gas"]  = df[gas_col]

df["totaal"] = df["gem_elek"] + df["gem_gas"] * 9.77
df["ratio"]  = df["gem_gas"] / df["gem_elek"].replace(0, np.nan)
df_clean = df.dropna(subset=["gem_elek", "gem_gas", "totaal", "ratio"])
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

# ── Clustering ────────────────────────────────────────────────────────────────
features = ["gem_elek", "gem_gas", "totaal", "ratio"]
scaler = StandardScaler()
X = scaler.fit_transform(df_clean[features])
km = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
df_clean["cluster"] = km.fit_predict(X)

# ── KPI\'s bovenaan ────────────────────────────────────────────────────────────
col1, col2, col3, col4 = st.columns(4)
col1.metric("MKB-adressen", f"{len(df_clean):,}")
col2.metric("Clusters", k_clusters)
col3.metric("Gem. elektra", f"{df_clean[\'gem_elek\'].mean():,.0f} kWh")
col4.metric("Gem. gas", f"{df_clean[\'gem_gas\'].mean():,.0f} m³")

# ── Scatter plot ──────────────────────────────────────────────────────────────
st.subheader("🔍 Clusters visualiseren")
fig_scatter = px.scatter(
    df_clean, x="gem_elek", y="gem_gas",
    color="cluster", color_continuous_scale="Viridis",
    hover_name=sector_col,
    size="totaal", opacity=0.75,
    labels={"gem_elek": "Elektra (kWh/adres)", "gem_gas": "Gas (m3/adres)"},
    template="plotly_white"
)
st.plotly_chart(fig_scatter, use_container_width=True)

# ── Besparingspotentieel ──────────────────────────────────────────────────────
st.subheader("💰 Besparingspotentieel")
profiles = df_clean.groupby("cluster")[["gem_elek", "gem_gas", "totaal"]].mean()
bench_elek = profiles["gem_elek"].min()
bench_gas  = profiles["gem_gas"].min()

profiles["besparing_eur"] = (
    (profiles["gem_elek"] - bench_elek).clip(lower=0) * prijs_elek +
    (profiles["gem_gas"]  - bench_gas ).clip(lower=0) * prijs_gas
)

fig_bar = px.bar(profiles.reset_index(), x="cluster", y="besparing_eur",
                  color="besparing_eur", color_continuous_scale="RdYlGn_r",
                  labels={"cluster": "Cluster", "besparing_eur": "Besparing (€/adres)"},
                  template="plotly_white")
fig_bar.update_layout(yaxis_tickprefix="€", showlegend=False)
st.plotly_chart(fig_bar, use_container_width=True)

# ── Tabel ─────────────────────────────────────────────────────────────────────
st.subheader("📋 Cluster overzicht")
st.dataframe(
    profiles.style.format({
        "gem_elek": "{:,.0f} kWh",
        "gem_gas":  "{:,.0f} m³",
        "totaal":   "{:,.0f} kWh-eq",
        "besparing_eur": "€{:,.0f}"
    }).background_gradient(subset=["besparing_eur"], cmap="RdYlGn_r"),
    use_container_width=True
)

st.caption("Besparing t.o.v. meest efficiënte cluster · CBS open data · Project 003")
'''

with open('app.py', 'w') as f:
    f.write(streamlit_code)

print('✅ Streamlit app opgeslagen als app.py')
print('\nOm te starten: streamlit run app.py')

## 7. Samenvatting & Conclusies

In [ ]:
# ─── Eindoverzicht ────────────────────────────────────────────────────────────
print('='*60)
print('📋 PROJECT 003 — CONCLUSIES')
print('='*60)
print(f'\n✅ {K_FINAL} clusters geïdentificeerd in MKB-energieverbruik')
print(f'✅ Silhouette score: {silhouette_score(X_scaled, df_mkb_clean["cluster"]):.3f}')

totale_besparing = df_savings['besparing_totaal_eur'].sum()
print(f'\n💰 Totaal besparingspotentieel: €{totale_besparing:,.0f}')
print(f'   Gebaseerd op {len(df_mkb_clean)} MKB-adressen')
print(f'   Benchmark cluster: Cluster {benchmark_cluster}')

print('\n✅ Gegenereerde bestanden:')
print('   energy_clusters.html  → K-means clusters PCA visualisatie')
print('   energy_savings.html   → Besparingspotentieel per cluster')
print('   app.py                → Streamlit dashboard')
print('\n🚀 Klaar voor portfolio!')

---
## 📁 Gegenereerde bestanden

| Bestand | Beschrijving |
|---|---|
| `energy_clusters.html` | K-means clusters — PCA scatter (Plotly) |
| `energy_savings.html` | Besparingspotentieel per cluster (Plotly) |
| `app.py` | Streamlit dashboard — interactief MKB-advies |

## 🔗 Databronnen
- [CBS StatLine 84901NED](https://opendata.cbs.nl/statline/portal.html?_la=nl&_catalog=CBS&tableId=84901NED) — Aardgas- en elektriciteitslevering aan bedrijven
- [CBS StatLine 83989NED](https://opendata.cbs.nl/statline/portal.html?_la=nl&_catalog=CBS&tableId=83989NED) — Energiebalans per sector

## ▶️ Streamlit starten
```bash
pip install streamlit cbsodata plotly scikit-learn
streamlit run app.py
```

---
*Project 003 | Ishan Sewnandan | Rotterdam, 2025*